In [46]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ai-tech-market-risk"
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [
    RAW_DATA_DIR,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    MODEL_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)
print("Models:", MODEL_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/ai-tech-market-risk
Raw data: /content/drive/MyDrive/ai-tech-market-risk/data/raw
Processed data: /content/drive/MyDrive/ai-tech-market-risk/data/processed
Models: /content/drive/MyDrive/ai-tech-market-risk/models


In [47]:
import pandas as pd
import numpy as np

In [11]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ai-tech-market-risk"
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [13]:
market_data = pd.read_csv(
    RAW_DATA_DIR / "market_data.csv",
    parse_dates=["Date"]
)

market_data.head()

,Date,Close,High,Low,Open,Volume,Ticker,Return_1D
0,2018-01-02,10.98,11.02,10.34,10.42,44146300,AMD,NaN
1,2018-01-03,11.55,12.14,11.36,11.61,154066700,AMD,0.051913
2,2018-01-04,12.12,12.43,11.97,12.10,109503000,AMD,0.049351
3,2018-01-05,11.88,12.22,11.66,12.19,63808900,AMD,-0.019802
4,2018-01-08,12.28,12.30,11.85,12.01,63346000,AMD,0.033670


In [14]:
print(market_data.shape)
print(market_data.dtypes)

(17368, 8)
Date         datetime64[ns]
Close               float64
High                float64
Low                 float64
Open                float64
Volume                int64
Ticker               object
Return_1D           float64
dtype: object


In [15]:
STOCKS = [
    "NVDA",
    "AMD",
    "MSFT",
    "GOOGL",
    "META",
]

BENCHMARKS = [
    "SPY",
    "QQQ",
    "SMH",
]

TICKERS = STOCKS + BENCHMARKS

In [16]:
market_features = market_data.copy()

for period in [5, 10, 20]:
    market_features[f"Return_{period}D"] = (
        market_features
        .groupby("Ticker")["Close"]
        .transform(
            lambda x: x.pct_change(
                periods=period,
                fill_method=None
            )
        )
    )

In [17]:
for window in [5, 10, 20]:
    market_features[f"Volatility_{window}D"] = (
        market_features
        .groupby("Ticker")["Return_1D"]
        .transform(
            lambda x: x.rolling(window).std()
        )
    )

In [18]:
market_features["Volume_Change_1D"] = (
    market_features
    .groupby("Ticker")["Volume"]
    .transform(
        lambda x: x.pct_change(fill_method=None)
    )
)

In [19]:
market_features["Volume_MA_20D"] = (
    market_features
    .groupby("Ticker")["Volume"]
    .transform(
        lambda x: x.rolling(20).mean()
    )
)

In [20]:
market_features["Relative_Volume_20D"] = (
    market_features["Volume"]
    / market_features["Volume_MA_20D"]
)

In [21]:
for window in [5, 20]:
    ma = (
        market_features
        .groupby("Ticker")["Close"]
        .transform(
            lambda x: x.rolling(window).mean()
        )
    )

    market_features[f"Price_vs_MA_{window}D"] = (
        market_features["Close"] / ma - 1
    )

In [22]:
benchmark_columns = [
    "Date",
    "Ticker",
    "Return_1D",
    "Return_5D"
]

benchmark_data = (
    market_features[
        market_features["Ticker"].isin(BENCHMARKS)
    ][benchmark_columns]
    .copy()
)

In [23]:
benchmark_1d = (
    benchmark_data
    .pivot(
        index="Date",
        columns="Ticker",
        values="Return_1D"
    )
    .rename(
        columns={
            "SPY": "SPY_Return_1D",
            "QQQ": "QQQ_Return_1D",
            "SMH": "SMH_Return_1D",
        }
    )
)

benchmark_5d = (
    benchmark_data
    .pivot(
        index="Date",
        columns="Ticker",
        values="Return_5D"
    )
    .rename(
        columns={
            "SPY": "SPY_Return_5D",
            "QQQ": "QQQ_Return_5D",
            "SMH": "SMH_Return_5D",
        }
    )
)

In [24]:
benchmark_features = (
    benchmark_1d
    .join(benchmark_5d)
    .reset_index()
)

benchmark_features.head()

Ticker,Date,QQQ_Return_1D,SMH_Return_1D,SPY_Return_1D,QQQ_Return_5D,SMH_Return_5D,SPY_Return_5D
0,2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-03,0.009717,0.014148,0.006325,NaN,NaN,NaN
2,2018-01-04,0.001749,0.005109,0.004215,NaN,NaN,NaN
3,2018-01-05,0.010043,0.006451,0.006664,NaN,NaN,NaN
4,2018-01-08,0.003891,0.006895,0.001829,NaN,NaN,NaN


In [25]:
stock_data = (
    market_features[
        market_features["Ticker"].isin(STOCKS)
    ]
    .copy()
)

In [26]:
stock_data = stock_data.merge(
    benchmark_features,
    on="Date",
    how="left"
)

In [27]:
stock_data["Excess_vs_QQQ_1D"] = (
    stock_data["Return_1D"]
    - stock_data["QQQ_Return_1D"]
)

stock_data["Excess_vs_SMH_1D"] = (
    stock_data["Return_1D"]
    - stock_data["SMH_Return_1D"]
)

In [28]:
stock_data["Future_Return_5D"] = (
    stock_data
    .groupby("Ticker")["Close"]
    .transform(
        lambda x: x.shift(-5) / x - 1
    )
)

In [29]:
stock_data["Large_Move_5D"] = np.where(
    stock_data["Future_Return_5D"].isna(),
    np.nan,
    (
        stock_data["Future_Return_5D"].abs()
        > 0.03
    ).astype(int)
)

In [30]:
stock_data[
    stock_data["Ticker"] == "NVDA"
][
    [
        "Date",
        "Close",
        "Future_Return_5D",
        "Large_Move_5D",
    ]
].tail(10)

,Date,Close,Future_Return_5D,Large_Move_5D
10845,2026-08-10,217.550003,0.034291,1.0
10846,2026-08-11,217.500000,0.010299,0.0
10847,2026-08-12,224.089996,-0.029140,0.0
10848,2026-08-13,225.300003,-0.037506,1.0
10849,2026-08-14,225.160004,-0.046367,1.0
10850,2026-08-17,225.009995,NaN,NaN
10851,2026-08-18,219.740005,NaN,NaN
10852,2026-08-19,217.559998,NaN,NaN
10853,2026-08-20,216.850006,NaN,NaN
10854,2026-08-21,214.720001,NaN,NaN


In [31]:
FEATURE_COLUMNS = [
    "Return_1D",
    "Return_5D",
    "Return_10D",
    "Return_20D",

    "Volatility_5D",
    "Volatility_10D",
    "Volatility_20D",

    "Volume_Change_1D",
    "Relative_Volume_20D",

    "Price_vs_MA_5D",
    "Price_vs_MA_20D",

    "SPY_Return_1D",
    "QQQ_Return_1D",
    "SMH_Return_1D",

    "SPY_Return_5D",
    "QQQ_Return_5D",
    "SMH_Return_5D",

    "Excess_vs_QQQ_1D",
    "Excess_vs_SMH_1D",
]

In [32]:
ml_data = stock_data[
    [
        "Date",
        "Ticker",
        "Close",
        *FEATURE_COLUMNS,
        "Future_Return_5D",
        "Large_Move_5D",
    ]
].copy()

In [33]:
ml_data = ml_data.dropna(
    subset=FEATURE_COLUMNS + ["Large_Move_5D"]
).copy()

In [34]:
ml_data["Large_Move_5D"] = (
    ml_data["Large_Move_5D"].astype(int)
)

In [35]:
ml_data = (
    ml_data
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

In [36]:
print("Rows:", len(ml_data))
print("Features:", len(FEATURE_COLUMNS))

Rows: 10730
Features: 19


In [37]:
print(
    ml_data[
        FEATURE_COLUMNS
        + ["Large_Move_5D"]
    ].isna().sum()
)

Return_1D              0
Return_5D              0
Return_10D             0
Return_20D             0
Volatility_5D          0
Volatility_10D         0
Volatility_20D         0
Volume_Change_1D       0
Relative_Volume_20D    0
Price_vs_MA_5D         0
Price_vs_MA_20D        0
SPY_Return_1D          0
QQQ_Return_1D          0
SMH_Return_1D          0
SPY_Return_5D          0
QQQ_Return_5D          0
SMH_Return_5D          0
Excess_vs_QQQ_1D       0
Excess_vs_SMH_1D       0
Large_Move_5D          0
dtype: int64


In [38]:
numeric_check = ml_data[
    FEATURE_COLUMNS
    + ["Future_Return_5D"]
]

print(
    "Infinite values:",
    np.isinf(numeric_check.to_numpy()).sum()
)

Infinite values: 0


In [39]:
target_distribution = (
    ml_data["Large_Move_5D"]
    .value_counts()
    .sort_index()
)

target_percentage = (
    ml_data["Large_Move_5D"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print("Counts:")
print(target_distribution)

print("\nPercentages:")
print(target_percentage)

Counts:
Large_Move_5D
0    5186
1    5544
Name: count, dtype: int64

Percentages:
Large_Move_5D
0    48.33178
1    51.66822
Name: proportion, dtype: float64


In [40]:
ticker_target_rate = (
    ml_data
    .groupby("Ticker")["Large_Move_5D"]
    .agg(
        observations="count",
        large_moves="sum",
        large_move_rate="mean",
    )
)

ticker_target_rate["large_move_rate"] *= 100

ticker_target_rate

,observations,large_moves,large_move_rate
Ticker,,,
AMD,2146,1428,66.542404
GOOGL,2146,908,42.311277
META,2146,1083,50.465983
MSFT,2146,764,35.601118
NVDA,2146,1361,63.420317


In [41]:
example = (
    ml_data[
        ml_data["Ticker"] == "NVDA"
    ]
    .iloc[500]
)

example

,2504
Date,2020-01-28 00:00:00
Ticker,NVDA
Close,6.164164
Return_1D,0.032348
Return_5D,0.000121
Return_10D,-0.015914
Return_20D,0.046861
Volatility_5D,0.02746
Volatility_10D,0.020029
Volatility_20D,0.017748


In [42]:
ticker = example["Ticker"]
date = example["Date"]

position = stock_data[
    stock_data["Ticker"] == ticker
].reset_index(drop=True)

row_index = position[
    position["Date"] == date
].index[0]

current_price = position.loc[row_index, "Close"]
future_price = position.loc[row_index + 5, "Close"]

manual_future_return = (
    future_price / current_price - 1
)

print("Date:", date)
print("Current price:", current_price)
print("Price 5 trading days later:", future_price)

print(
    "Manual future return:",
    manual_future_return
)

print(
    "Stored future return:",
    example["Future_Return_5D"]
)

Date: 2020-01-28 00:00:00
Current price: 6.164163589477539
Price 5 trading days later: 6.143281936645508
Manual future return: -0.0033875890100770745
Stored future return: -0.0033875890100770745


In [48]:
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

ml_data.to_csv(
    PROCESSED_DATA_DIR / "ml_features.csv",
    index=False
)

print(
    "Saved:",
    PROCESSED_DATA_DIR / "ml_features.csv"
)

Saved: /content/drive/MyDrive/ai-tech-market-risk/data/processed/ml_features.csv


In [49]:
raw_file = RAW_DATA_DIR / "market_data.csv"

print("Exists:", raw_file.exists())
print("Size:", raw_file.stat().st_size, "bytes")

Exists: True
Size: 2066725 bytes
